# 07 — Ajuste fino do detector antigo com HomeObjects + SUN RGB-D

Execute no **Google Colab com GPU T4**, após concluir o notebook 06.
O treinamento começa no `best.pt` antigo. É um novo experimento (`resume=False`),
com dados mistos, taxa de aprendizado menor e HomeObjects mantido para reduzir
o esquecimento das classes atuais. A arquitetura e os 12 IDs permanecem iguais.

O objetivo é melhorar a precisão de detecção e dos alertas, acompanhando também
Recall e omissões de alta prioridade. Mais imagens não garantem melhoria.
Esta etapa compara validação, não consulta TEST para ajustar parâmetros ou
selecionar o candidato. A avaliação final usa o notebook 05d no mesmo ambiente
local para ambos os modelos. A V2 e a saída em texto permanecem as atuais.


## 1. Instalar a versão definida e ativar a GPU

Usamos a versão do Ultralytics das comparações locais anteriores. Se o Colab pedir
reinício depois de instalar, reinicie e execute novamente a partir desta etapa.


In [1]:
%pip install -q ultralytics==8.4.150 pyyaml pandas pillow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.8/96.8 kB 4.4 MB/s eta 0:00:00


In [2]:
import sys
import json
import hashlib
import platform
import shutil
import zipfile
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import yaml
import torch
import ultralytics
from ultralytics import YOLO
from IPython.display import display

if not torch.cuda.is_available():
    raise RuntimeError("No Colab, selecione Ambiente de execução → Alterar tipo → GPU T4 e reconecte.")
if ultralytics.__version__ != "8.4.150":
    raise RuntimeError("Reinicie o ambiente para carregar ultralytics==8.4.150.")
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("GPU:", torch.cuda.get_device_name(0))


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Python: 3.13.15
PyTorch: 2.11.0+cu130
Ultralytics: 8.4.150
GPU: Tesla T4


## 2. Montar o Drive e conferir as entradas

No Meu Drive, dentro de `TCC_Visao_Computacional`, use:

| Arquivo | Caminho |
|---|---|
| ZIP gerado no 06 | `datasets/homeobjects_sunrgbd_validado.zip` |
| Modelo antigo | `modelos/yolov8n_baseline/best.pt` |

Os resultados vão para uma pasta nova com data/hora. Não substituem o baseline.


In [3]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [7]:
DRIVE_TCC = Path("/content/drive/MyDrive/TCC_Visao_Computacional")
ZIP_DATASET = DRIVE_TCC / "datasets/homeobjects_sunrgbd_validado.zip"
MODELO_ANTIGO_DRIVE = DRIVE_TCC / "modelos/yolov8n_baseline/best.pt"
SHA_MODELO_ANTIGO = "4170653e0c0a604f85b6747e42b3fd2ff5ddb765ee4762915dfc9f3d33d322d9"
NOME_EXPERIMENTO = "yolov8n_baseline_sunrgbd_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
PASTA_TRABALHO = Path("/content/tcc_sunrgbd") / NOME_EXPERIMENTO
PASTA_RESULTADOS = DRIVE_TCC / "resultados" / NOME_EXPERIMENTO
PASTA_RUN = PASTA_RESULTADOS / "treinamento"

# Configuração inicial definida antes de consultar o TEST.
EPOCHS = 30
PATIENCE = 10
BATCH = 16
IMGSZ = 640
SEED = 42
WORKERS = 2
LR0 = 0.0003

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with Path(caminho).open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

for p in [ZIP_DATASET, MODELO_ANTIGO_DRIVE]:
    if not p.is_file():
        raise FileNotFoundError(f"Arquivo ausente no Drive: {p}")
if sha256_arquivo(MODELO_ANTIGO_DRIVE) != SHA_MODELO_ANTIGO:
    raise RuntimeError("O arquivo do Drive não corresponde ao modelo antigo avaliado. Copie o best.pt de models/yolov8n_baseline.")
PASTA_RESULTADOS.mkdir(parents=True, exist_ok=False)
PASTA_TRABALHO.mkdir(parents=True, exist_ok=False)
MODELO_INICIAL = PASTA_TRABALHO / "baseline_inicial.pt"
shutil.copy2(MODELO_ANTIGO_DRIVE, MODELO_INICIAL)
print("Modelo inicial confirmado:", MODELO_INICIAL)
print("Resultados:", PASTA_RESULTADOS)


Modelo inicial confirmado: /content/tcc_sunrgbd/yolov8n_baseline_sunrgbd_20261013_120842_048419/baseline_inicial.pt
Resultados: /content/drive/MyDrive/TCC_Visao_Computacional/resultados/yolov8n_baseline_sunrgbd_20261013_120842_048419


## 3. Extrair e conferir o pacote completo

Cada arquivo do manifesto é conferido por SHA-256. Também verificamos os 12 IDs,
as contagens e a ausência de imagens/labels adicionais fora do manifesto.
O YAML do Colab é uma cópia com a raiz local corrigida.


In [8]:
destino = PASTA_TRABALHO.resolve()
with zipfile.ZipFile(ZIP_DATASET) as pacote:
    nomes_zip = pacote.namelist()
    if len(nomes_zip) != len(set(nomes_zip)):
        raise ValueError("ZIP contém nomes repetidos.")
    for membro in pacote.infolist():
        final = (destino / membro.filename).resolve()
        if final != destino and destino not in final.parents:
            raise ValueError("ZIP contém caminho fora da pasta de trabalho.")
    pacote.extractall(destino)

manifesto = json.loads((PASTA_TRABALHO / "verificacao_preparacao.json").read_text(encoding="utf-8"))
for item in manifesto["arquivos"]:
    arquivo = (PASTA_TRABALHO / item["arquivo"]).resolve()
    if destino not in arquivo.parents or not arquivo.is_file() or sha256_arquivo(arquivo) != item["sha256"]:
        raise RuntimeError(f"Arquivo ausente ou alterado: {item['arquivo']}")
if manifesto["test"]["imagens"] != 403:
    raise RuntimeError("O pacote não preserva o TEST de 403 imagens.")
RAIZ_DATASET = PASTA_TRABALHO / manifesto["dataset_dir"]
dados_yaml = yaml.safe_load((RAIZ_DATASET / "data.yaml").read_text(encoding="utf-8-sig"))
NOMES = {int(k): str(v) for k, v in dados_yaml["names"].items()}
ESPERADAS = dict(enumerate(["bed", "sofa", "chair", "table", "lamp", "tv", "laptop", "wardrobe", "window", "door", "potted plant", "photo frame"]))
if NOMES != ESPERADAS or NOMES != {int(k): v for k, v in manifesto["classes"].items()}:
    raise RuntimeError("Classes/IDs incompatíveis com o baseline.")
verificacao = []
for split in ["train", "val", "test"]:
    for tipo in ["images", "labels"]:
        prefixo = f"{manifesto['dataset_dir']}/{tipo}/{split}/"
        esperados = {i["arquivo"] for i in manifesto["arquivos"] if i["arquivo"].startswith(prefixo)}
        reais = {p.relative_to(PASTA_TRABALHO).as_posix() for p in (RAIZ_DATASET / tipo / split).rglob("*") if p.is_file()}
        if reais != esperados or len(reais) != manifesto["imagens_por_split"][split]:
            raise RuntimeError(f"Arquivos adicionais/ausentes ou contagem diferente: {tipo}/{split}.")
    verificacao.append({"split": split, "imagens": manifesto["imagens_por_split"][split]})
dados_yaml["path"] = RAIZ_DATASET.as_posix()
YAML_MISTO = RAIZ_DATASET / "data_colab.yaml"
YAML_MISTO.write_text(yaml.safe_dump(dados_yaml, sort_keys=False, allow_unicode=True), encoding="utf-8")
sys.path.insert(0, str(PASTA_TRABALHO / "codigo"))
from src.validacao_alertas import carregar_home_val, avaliar_alertas
import src.politica_alertas as politica
REGISTROS_HOME_VAL = carregar_home_val(RAIZ_DATASET, NOMES)
lista_home_val = PASTA_TRABALHO / "home_val.txt"
lista_home_val.write_text("\n".join(str(r["imagem"]) for r in REGISTROS_HOME_VAL) + "\n", encoding="utf-8")
# Uma avaliação de detecção somente na validação Home, separada da validação mista.
YAML_HOME_VAL = PASTA_TRABALHO / "data_home_val.yaml"
home_yaml = {**dados_yaml, "val": str(lista_home_val)}
YAML_HOME_VAL.write_text(yaml.safe_dump(home_yaml, allow_unicode=True, sort_keys=False), encoding="utf-8")
display(pd.DataFrame(verificacao))
print("Validação Home:", len(REGISTROS_HOME_VAL), "imagens")
shutil.copy2(PASTA_TRABALHO / "verificacao_preparacao.json", PASTA_RESULTADOS / "verificacao_preparacao.json")
shutil.copy2(PASTA_TRABALHO / "resumo_preparacao_sunrgbd.json", PASTA_RESULTADOS / "resumo_preparacao_sunrgbd.json")


,split,imagens
0,train,3970
1,val,993
2,test,403


Validação Home: 456 imagens


PosixPath('/content/drive/MyDrive/TCC_Visao_Computacional/resultados/yolov8n_baseline_sunrgbd_20261013_120842_048419/resumo_preparacao_sunrgbd.json')

## 4. Registrar configuração e avaliar o antigo na validação Home

O controle e o candidato usam o mesmo conjunto, GPU e parâmetros. As métricas
dos alertas usam confiança 0,25, NMS IoU 0,70, correspondência de mesma classe
com IoU ≥ 0,50 e alertas de prioridade média/alta. As métricas mAP usam 0,001.


In [9]:
CONFIGURACAO = {
    "experimento": NOME_EXPERIMENTO, "modelo_base_sha256": SHA_MODELO_ANTIGO,
    "zip_sha256": sha256_arquivo(ZIP_DATASET), "epochs": EPOCHS,
    "patience": PATIENCE, "batch": BATCH, "imgsz": IMGSZ, "seed": SEED,
    "workers": WORKERS, "optimizer": "AdamW", "lr0": LR0,
    "resume": False, "mosaic": 0.15, "close_mosaic": 5,
    "python": platform.python_version(), "ultralytics": ultralytics.__version__,
    "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0),
    "test_sha256": manifesto["test"]["sha256"],
    "politica_sha256": sha256_arquivo(politica.__file__),
    "home_val_imagens": len(REGISTROS_HOME_VAL),
    "criterio_checkpoint": "best.pt por fitness de validação mista do Ultralytics",
    "criterio_triagem": "ganho de precisão de alertas com Recall preservado, na validação Home",
}
(PASTA_RESULTADOS / "configuracao_experimento.json").write_text(
    json.dumps(CONFIGURACAO, ensure_ascii=False, indent=2), encoding="utf-8")

def avaliar_detector_val(modelo, nome):
    r = modelo.val(data=str(YAML_HOME_VAL), split="val", imgsz=IMGSZ,
                   batch=BATCH, device=0, conf=0.001, iou=0.70, max_det=300,
                   plots=True, project=str(PASTA_RESULTADOS), name=nome, exist_ok=False)
    return {"precision_detector": float(r.box.mp), "recall_detector": float(r.box.mr),
            "map50": float(r.box.map50), "map50_95": float(r.box.map)}

antigo = YOLO(str(MODELO_INICIAL))
if {int(k): str(v) for k, v in antigo.names.items()} != NOMES:
    raise RuntimeError("Modelo antigo tem IDs diferentes dos labels.")
metricas_antigo = {**avaliar_detector_val(antigo, "baseline_home_val"),
                   **avaliar_alertas(antigo, REGISTROS_HOME_VAL, NOMES)}
(PASTA_RESULTADOS / "metricas_baseline_val.json").write_text(json.dumps(metricas_antigo, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(metricas_antigo, ensure_ascii=False, indent=2))
del antigo
torch.cuda.empty_cache()


Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,007,988 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2490.6±566.2 MB/s, size: 180.1 KB)
val: Scanning /content/tcc_sunrgbd/yolov8n_baseline_sunrgbd_20261013_120842_048419/homeobjects_yolo/labels/val/raw_train... 456 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 456/456 2.2Kit/s 0.2s
val: New cache created: /content/tcc_sunrgbd/yolov8n_baseline_sunrgbd_20261013_120842_048419/homeobjects_yolo/labels/val/raw_train.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 2.7it/s 10.7s
                   all        456       3682      0.883      0.765      0.859      0.664
                   bed         29         29      0.907      0.793      0.879      0.719
                  sofa        294        420      0.912      0.921      0.952      0.792
                 

## 5. Ajuste fino a partir do best.pt antigo

30 épocas, parada após 10 sem melhora, AdamW com `lr0=0.0003` e aumentos
moderados. `resume=False` inicia uma nova agenda de aprendizado, mantendo os
pesos aprendidos. Não usa o último treino como uma retomada de épocas.

Os pesos e logs são gravados diretamente na pasta nova do Drive, inclusive
`last.pt` por época. Se a memória da GPU for insuficiente, reduza BATCH para 8
antes de iniciar outro experimento e mantenha o valor registrado na configuração.


In [10]:
modelo = YOLO(str(MODELO_INICIAL))
resultado_treino = modelo.train(
    data=str(YAML_MISTO), epochs=EPOCHS, patience=PATIENCE,
    imgsz=IMGSZ, batch=BATCH, device=0, workers=WORKERS,
    seed=SEED, deterministic=True, optimizer="AdamW", lr0=LR0,
    lrf=0.10, cos_lr=True, warmup_epochs=2.0, warmup_bias_lr=0.0,
    weight_decay=0.0005, mosaic=0.15, close_mosaic=5,
    mixup=0.0, scale=0.20, translate=0.10, fliplr=0.50,
    resume=False, pretrained=True, save=True, save_period=5,
    plots=True, cache=False, project=str(PASTA_RESULTADOS),
    name="treinamento", exist_ok=False,
)
PASTA_RUN = Path(modelo.trainer.save_dir)
BEST = PASTA_RUN / "weights/best.pt"
if not BEST.is_file():
    raise FileNotFoundError(f"best.pt não produzido: {BEST}")
shutil.copy2(BEST, PASTA_RESULTADOS / "best.pt")
print("Novo candidato preservado:", PASTA_RESULTADOS / "best.pt")
del modelo
torch.cuda.empty_cache()


New https://pypi.org/project/ultralytics/8.4.176 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/tcc_sunrgbd/yolov8n_baseline_sunrgbd_20261013_120842_048419/homeobjects_yolo/data_colab.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0003, 

## 6. Comparar com o antigo na mesma validação

O `best.pt` é escolhido pelo critério padrão do Ultralytics na validação mista;
não é necessariamente a época de maior precisão dos alertas. A comparação
abaixo verifica se ele atende ao objetivo do projeto. Não aumente a confiança
com base no TEST para fazer a precisão parecer maior.


In [11]:
candidato = YOLO(str(PASTA_RESULTADOS / "best.pt"))
metricas_candidato = {**avaliar_detector_val(candidato, "candidato_home_val"),
                      **avaliar_alertas(candidato, REGISTROS_HOME_VAL, NOMES)}
(PASTA_RESULTADOS / "metricas_candidato_val.json").write_text(json.dumps(metricas_candidato, indent=2, ensure_ascii=False), encoding="utf-8")
chaves = ["precision_detector", "recall_detector", "map50", "map50_95",
          "precision_alertas", "recall_alertas", "f1_alertas", "fn", "fp", "alta_sem_alerta"]
comparacao = pd.DataFrame([
    {"metrica": k, "antigo": metricas_antigo[k], "candidato": metricas_candidato[k],
     "diferenca": None if metricas_antigo[k] is None or metricas_candidato[k] is None else metricas_candidato[k] - metricas_antigo[k]}
    for k in chaves
])
display(comparacao)
comparacao.to_csv(PASTA_RESULTADOS / "comparacao_validacao.csv", index=False, encoding="utf-8-sig")
pa, pn = metricas_antigo["precision_alertas"], metricas_candidato["precision_alertas"]
ra, rn = metricas_antigo["recall_alertas"], metricas_candidato["recall_alertas"]
promissor = all(v is not None for v in [pa, pn, ra, rn]) and pn > pa and rn >= ra and metricas_candidato["alta_sem_alerta"] <= metricas_antigo["alta_sem_alerta"]
triagem = {"promissor_na_validacao": bool(promissor),
           "criterio": "Precisão dos alertas maior, Recall não menor e omissões de alta prioridade não maiores, em Home VAL.",
           "observacao": "Triagem exploratória; não é teste de significância nem validação de risco real. Histórico do baseline pode incluir parte do Home VAL atual.",
           "proximo_passo": "Comparar antigo e candidato no 05d local, com TEST e parâmetros preservados; não substituir o baseline automaticamente."}
(PASTA_RESULTADOS / "triagem_validacao.json").write_text(json.dumps(triagem, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(triagem, ensure_ascii=False, indent=2))
print("Resultados:", PASTA_RESULTADOS)


Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,007,988 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2897.4±1496.4 MB/s, size: 163.5 KB)
val: Scanning /content/tcc_sunrgbd/yolov8n_baseline_sunrgbd_20261013_120842_048419/homeobjects_yolo/labels/val/raw_train... 456 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 456/456 2.0Kit/s 0.2s
val: New cache created: /content/tcc_sunrgbd/yolov8n_baseline_sunrgbd_20261013_120842_048419/homeobjects_yolo/labels/val/raw_train.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 3.4it/s 8.5s
                   all        456       3682        0.8      0.736      0.819      0.616
                   bed         29         29      0.831      0.793      0.838      0.715
                  sofa        294        420      0.856      0.886      0.927      0.746
                 

,metrica,antigo,candidato,diferenca
0,precision_detector,0.882927,0.800123,-0.082804
1,recall_detector,0.765052,0.735975,-0.029077
2,map50,0.859207,0.819436,-0.039772
3,map50_95,0.664002,0.616436,-0.047566
4,precision_alertas,0.783646,0.746674,-0.036972
5,recall_alertas,0.899023,0.877524,-0.021498
6,f1_alertas,0.837379,0.806828,-0.030550
7,fn,155.000000,188.000000,33.000000
8,fp,381.000000,457.000000,76.000000
9,alta_sem_alerta,24.000000,28.000000,4.000000


{
  "promissor_na_validacao": false,
  "criterio": "Precisão dos alertas maior, Recall não menor e omissões de alta prioridade não maiores, em Home VAL.",
  "observacao": "Triagem exploratória; não é teste de significância nem validação de risco real. Histórico do baseline pode incluir parte do Home VAL atual.",
  "proximo_passo": "Comparar antigo e candidato no 05d local, com TEST e parâmetros preservados; não substituir o baseline automaticamente."
}
Resultados: /content/drive/MyDrive/TCC_Visao_Computacional/resultados/yolov8n_baseline_sunrgbd_20261013_120842_048419


## 7. Avaliação final no Windows

Baixe `best.pt` desta pasta nova e salve em:

`models/yolov8n_baseline_sunrgbd_01/best.pt`

No **05d local**, mantenha a base TEST original `data/processed/homeobjects_yolo`
e os parâmetros já utilizados. Mude somente:

```python
CAMINHO_MODELO = RAIZ_PROJETO / "models/yolov8n_baseline_sunrgbd_01/best.pt"
```

Reavalie também o antigo na mesma instalação local se as bibliotecas mudarem.
O 04 V2 demonstra mensagens em texto, enquanto o 05d mede a comparação.

Arquivos para documentar o experimento 07: `comparacao_validacao.csv`,
`configuracao_experimento.json` e `triagem_validacao.json`.
Os resultados do notebook 05d permitem comparação com a referência anterior.
O candidato só passa a ser usado após essa análise; o antigo fica preservado.

O TEST já foi consultado em experimentos anteriores e funciona como benchmark
fixo de comparação. Não o usamos para ajustar classes, confiança ou parâmetros.
Uma conclusão final de generalização pede uma base independente ainda não usada.
